# 🏢 Vendor & Third-Party Risk Management (TPRM)
## Third-Party Risk Assessment Register — Enterprise Risk Management

**Objective:** Build a third-party risk assessment register for 40 simulated vendors — scoring each across five risk dimensions (financial stability, cybersecurity posture, regulatory compliance, geographic risk, concentration risk), tiering vendors into Critical/High/Medium/Low risk categories, generating tier-specific due diligence checklists, and producing an operational risk dashboard for procurement and enterprise risk teams.

**Frameworks Applied:** NIST Cybersecurity Framework · ISO 31000 Risk Management · OCC Third-Party Risk Guidance · Basel Committee Outsourcing Principles  
**Tools:** Python · Pandas · Matplotlib · Seaborn · Rule-Based Risk Scoring Engine

> *All vendor names, financial figures, and risk data are entirely synthetic and created solely for portfolio demonstration purposes.*

---

## 1. Setup & Vendor Register Generation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta

In [ ]:

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')

RISK_COLORS = {'Low':'#3BAB6F','Medium':'#F39C12','High':'#E8834D','Critical':'#E74C3C'}
PALETTE = ['#2D6A9F','#E74C3C','#3BAB6F','#F39C12','#9B59B6','#1ABC9C','#E8834D','#34495E']

np.random.seed(42)

vendor_categories = ['IT Services','Cloud Infrastructure','Payment Processing','Logistics',
                     'Professional Services','Facilities Management','Marketing Agency',
                     'HR Services','Manufacturing Supplier','Telecommunications',
                     'Data Analytics','Legal Services','Office Supplies','Catering',
                     'Security Services']

countries = ['Kenya','United States','United Kingdom','India','Germany','Singapore',
             'South Africa','Nigeria','China','Russia','Brazil','UAE','Philippines',
             'Vietnam','Mexico']

HIGH_RISK_COUNTRIES = {'Russia','Nigeria','China','Vietnam'}

vendor_names = [
    'Apex Cloud Solutions','Meridian IT Services','Coastal Logistics Partners',
    'Sterling Payment Systems','Quantum Data Analytics','Highland Security Group',
    'Pinnacle Facilities Mgmt','Cascade HR Solutions','Northbridge Legal Associates',
    'Vertex Manufacturing Co','Crestview Marketing','Summit Telecom Services',
    'Riverside Catering Group','Oakwood Office Supplies','Falcon Cyber Defense',
    'Bluewave Cloud Hosting','Ironwood Consulting','Silverline Payments Ltd',
    'Eastgate Logistics','Western Star IT Group','Goldcoast Data Services',
    'Brightpath HR Outsourcing','Stonebridge Legal Partners','Redwood Manufacturing',
    'Skyline Marketing Agency','Centurion Security Solutions','Atlas Telecom Group',
    'Harbor Catering Services','Maple Office Solutions','Zenith Cyber Solutions',
    'Polaris Cloud Systems','Frontier IT Consulting','Delta Logistics Network',
    'Omega Payment Gateway','Nova Analytics Group','Titan Facilities Services',
    'Aurora HR Partners','Meridian Legal Group','Phoenix Manufacturing Ltd',
    'Pacific Marketing Solutions'
]

rows = []
for i, name in enumerate(vendor_names):
    category = np.random.choice(vendor_categories)
    country  = np.random.choice(countries, p=[0.30,0.12,0.08,0.08,0.06,0.05,
                                                0.05,0.05,0.05,0.04,0.04,0.03,
                                                0.02,0.02,0.01])
    annual_spend = round(np.random.lognormal(mean=11.5, sigma=1.3), -3)
    annual_spend = min(annual_spend, 4500000)

    # Financial stability indicators
    credit_rating   = np.random.choice(['AAA','AA','A','BBB','BB','B','CCC'],
                                       p=[0.05,0.10,0.20,0.25,0.20,0.15,0.05])
    years_in_business = np.random.randint(1, 35)
    financial_distress = np.random.choice([True, False], p=[0.12, 0.88])

    # Cybersecurity posture
    has_iso27001     = np.random.choice([True, False], p=[0.45, 0.55])
    has_soc2         = np.random.choice([True, False], p=[0.35, 0.65])
    recent_breach    = np.random.choice([True, False], p=[0.10, 0.90])
    data_access_level= np.random.choice(['None','Limited','Moderate','Extensive'],
                                        p=[0.20,0.30,0.30,0.20])

    # Regulatory compliance
    compliance_violations = np.random.poisson(0.4)
    has_required_licenses  = np.random.choice([True, False], p=[0.85, 0.15])
    gdpr_relevant          = np.random.choice([True, False], p=[0.40, 0.60])

    # Concentration risk
    is_sole_source   = np.random.choice([True, False], p=[0.20, 0.80])
    pct_of_category_spend = round(np.random.uniform(5, 95), 1)

    # Operational
    contract_critical = np.random.choice(['Critical','Important','Standard'],
                                         p=[0.20, 0.35, 0.45])
    sla_breaches_ytd   = np.random.poisson(1.2)

    rows.append({
        'Vendor_ID':            f'VND-{1000+i}',
        'Vendor_Name':          name,
        'Category':             category,
        'Country':              country,
        'Annual_Spend_USD':     annual_spend,
        'Credit_Rating':        credit_rating,
        'Years_In_Business':    years_in_business,
        'Financial_Distress':   financial_distress,
        'Has_ISO27001':         has_iso27001,
        'Has_SOC2':             has_soc2,
        'Recent_Breach':        recent_breach,
        'Data_Access_Level':    data_access_level,
        'Compliance_Violations':compliance_violations,
        'Has_Required_Licenses':has_required_licenses,
        'GDPR_Relevant':        gdpr_relevant,
        'Is_Sole_Source':       is_sole_source,
        'Pct_Category_Spend':   pct_of_category_spend,
        'Contract_Criticality': contract_critical,
        'SLA_Breaches_YTD':     sla_breaches_ytd,
        'Onboarded_Date':       datetime(2018,1,1) + timedelta(days=int(np.random.randint(0,2200))),
    })

df = pd.DataFrame(rows)
df['Is_High_Risk_Country'] = df['Country'].isin(HIGH_RISK_COUNTRIES)

print(f"Vendor Register: {len(df)} vendors")
print(f"Total annual spend: \${df['Annual_Spend_USD'].sum():,.0f}")
print(f"Categories: {df['Category'].nunique()}")
print(f"Countries: {df['Country'].nunique()}")
df[['Vendor_ID','Vendor_Name','Category','Country','Annual_Spend_USD',
   'Credit_Rating','Contract_Criticality']].head(8)


## 2. Five-Dimension Risk Scoring Engine

Each vendor is scored across five risk dimensions aligned to recognised TPRM frameworks:

| Dimension | Framework Reference | Trigger Factors |
|---|---|---|
| **Financial Stability** | ISO 31000 / OCC Guidance | Credit rating, years in business, financial distress signals |
| **Cybersecurity Posture** | NIST Cybersecurity Framework | Certifications (ISO 27001, SOC 2), breach history, data access level |
| **Regulatory Compliance** | Basel Outsourcing Principles | Compliance violations, licensing status, GDPR relevance |
| **Geographic Risk** | OCC Country Risk Guidance | Operating jurisdiction risk classification |
| **Concentration Risk** | ISO 31000 | Sole-source dependency, % of category spend |


In [ ]:
def score_financial(row):
    score = 0
    rating_risk = {'AAA':0,'AA':5,'A':10,'BBB':20,'BB':35,'B':50,'CCC':70}
    score += rating_risk.get(row['Credit_Rating'], 30)
    if row['Years_In_Business'] < 3:  score += 20
    elif row['Years_In_Business'] < 5: score += 10
    if row['Financial_Distress']:     score += 30
    return min(score, 100)

def score_cyber(row):
    score = 30  # baseline
    if row['Has_ISO27001']: score -= 15
    if row['Has_SOC2']:     score -= 10
    if row['Recent_Breach']: score += 45
    access_risk = {'None':0,'Limited':10,'Moderate':20,'Extensive':35}
    score += access_risk.get(row['Data_Access_Level'], 15)
    return max(min(score, 100), 0)

def score_compliance(row):
    score = row['Compliance_Violations'] * 20
    if not row['Has_Required_Licenses']: score += 40
    if row['GDPR_Relevant']:             score += 10
    return min(score, 100)

def score_geographic(row):
    return 65 if row['Is_High_Risk_Country'] else 15

def score_concentration(row):
    score = 0
    if row['Is_Sole_Source']: score += 40
    if row['Pct_Category_Spend'] > 70: score += 35
    elif row['Pct_Category_Spend'] > 50: score += 20
    elif row['Pct_Category_Spend'] > 30: score += 10
    return min(score, 100)

df['Score_Financial']     = df.apply(score_financial,     axis=1)
df['Score_Cyber']         = df.apply(score_cyber,         axis=1)
df['Score_Compliance']    = df.apply(score_compliance,    axis=1)
df['Score_Geographic']    = df.apply(score_geographic,    axis=1)
df['Score_Concentration'] = df.apply(score_concentration, axis=1)

# Weighted composite — weighted by contract criticality context
weights = {
    'Score_Financial':     0.25,
    'Score_Cyber':         0.30,
    'Score_Compliance':    0.20,
    'Score_Geographic':    0.10,
    'Score_Concentration': 0.15,
}
df['Composite_Risk_Score'] = sum(df[col]*w for col, w in weights.items()).round(1)

# Criticality multiplier — critical contracts amplify residual risk impact
crit_multiplier = {'Critical':1.15, 'Important':1.05, 'Standard':1.0}
df['Adjusted_Risk_Score'] = (df['Composite_Risk_Score'] *
                              df['Contract_Criticality'].map(crit_multiplier)).clip(0,100).round(1)

def assign_tier(score):
    if score >= 65: return 'Critical'
    elif score >= 45: return 'High'
    elif score >= 25: return 'Medium'
    else: return 'Low'

df['Risk_Tier'] = df['Adjusted_Risk_Score'].apply(assign_tier)

print("Risk Tier Distribution:")
print(df['Risk_Tier'].value_counts())
print()
print("Top 5 Highest Risk Vendors:")
print(df.nlargest(5,'Adjusted_Risk_Score')[
    ['Vendor_ID','Vendor_Name','Category','Risk_Tier','Adjusted_Risk_Score']].to_string(index=False))


## 3. Vendor Portfolio Risk Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('Third-Party Risk Management — Portfolio Overview', fontsize=15, fontweight='bold')

tier_order = ['Critical','High','Medium','Low']

# Risk tier distribution
ax = axes[0,0]
tc = df['Risk_Tier'].value_counts().reindex(tier_order)
bars = ax.bar(tier_order, tc.values, color=[RISK_COLORS[t] for t in tier_order],
              width=0.55, edgecolor='white')
ax.set_title('Vendor Risk Tier Distribution', fontweight='bold')
ax.set_ylabel('Number of Vendors')
for bar, val in zip(bars, tc.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            str(val), ha='center', fontsize=12, fontweight='bold')

# Spend by risk tier
ax = axes[0,1]
spend_tier = df.groupby('Risk_Tier')['Annual_Spend_USD'].sum().reindex(tier_order)
bars = ax.bar(tier_order, spend_tier.values, color=[RISK_COLORS[t] for t in tier_order],
              width=0.55, edgecolor='white')
ax.set_title('Total Annual Spend by Risk Tier', fontweight='bold')
ax.set_ylabel('Annual Spend (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
for bar, val in zip(bars, spend_tier.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30000,
            f'${val/1e6:.1f}M', ha='center', fontsize=9, fontweight='bold')

# Risk tier by category
ax = axes[0,2]
cat_tier = df.groupby(['Category','Risk_Tier']).size().unstack(fill_value=0)
cat_tier = cat_tier.reindex(columns=tier_order, fill_value=0)
cat_tier_top = cat_tier.loc[cat_tier.sum(axis=1).sort_values(ascending=False).head(8).index]
cat_tier_top.plot(kind='barh', stacked=True, ax=ax,
                  color=[RISK_COLORS[t] for t in tier_order], edgecolor='white')
ax.set_title('Risk Tier by Top 8 Vendor Categories', fontweight='bold')
ax.set_xlabel('Number of Vendors')
ax.legend(title='Risk Tier', fontsize=7, loc='lower right')

# Risk dimension averages
ax = axes[1,0]
dim_cols = ['Score_Financial','Score_Cyber','Score_Compliance','Score_Geographic','Score_Concentration']
dim_labels = ['Financial','Cyber','Compliance','Geographic','Concentration']
avg_dims = df[dim_cols].mean()
bars = ax.bar(dim_labels, avg_dims.values, color=PALETTE[:5], width=0.6, edgecolor='white')
ax.set_title('Average Risk Score by Dimension', fontweight='bold')
ax.set_ylabel('Average Score (0-100)')
for bar, val in zip(bars, avg_dims.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')

# Geographic risk map (countries)
ax = axes[1,1]
country_risk = df.groupby('Country').agg(
    Count=('Vendor_ID','count'),
    Avg_Risk=('Adjusted_Risk_Score','mean')
).reset_index().sort_values('Avg_Risk', ascending=False)
bar_colors = ['#E74C3C' if c in HIGH_RISK_COUNTRIES else '#2D6A9F' for c in country_risk['Country']]
ax.barh(country_risk['Country'][::-1], country_risk['Avg_Risk'][::-1],
        color=bar_colors[::-1], edgecolor='white')
ax.set_title('Avg Vendor Risk Score by Country\n(red = high-risk jurisdiction)', fontweight='bold')
ax.set_xlabel('Avg Adjusted Risk Score')

# Contract criticality vs risk tier
ax = axes[1,2]
crit_tier = df.groupby(['Contract_Criticality','Risk_Tier']).size().unstack(fill_value=0)
crit_tier = crit_tier.reindex(columns=tier_order, fill_value=0)
crit_tier = crit_tier.reindex(['Critical','Important','Standard'])
crit_tier.plot(kind='bar', ax=ax, color=[RISK_COLORS[t] for t in tier_order],
              edgecolor='white', width=0.65)
ax.set_title('Risk Tier by Contract Criticality', fontweight='bold')
ax.set_ylabel('Number of Vendors')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Risk Tier', fontsize=8)

plt.tight_layout()
plt.savefig
plt.show()


## 4. Risk Dimension Deep-Dive Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('Risk Dimension Deep-Dive Analysis', fontsize=15, fontweight='bold')

# Cybersecurity posture breakdown
ax = axes[0,0]
cyber_stats = {
    'Has ISO 27001': df['Has_ISO27001'].sum(),
    'Has SOC 2':      df['Has_SOC2'].sum(),
    'Recent Breach':  df['Recent_Breach'].sum(),
    'Neither Cert':   ((~df['Has_ISO27001']) & (~df['Has_SOC2'])).sum(),
}
bars = ax.bar(cyber_stats.keys(), cyber_stats.values(),
              color=['#3BAB6F','#2D6A9F','#E74C3C','#E8834D'], width=0.6, edgecolor='white')
ax.set_title('Cybersecurity Posture Indicators', fontweight='bold')
ax.set_ylabel('Number of Vendors')
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, cyber_stats.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            str(val), ha='center', fontsize=10, fontweight='bold')

# Data access level vs cyber score
ax = axes[0,1]
access_order = ['None','Limited','Moderate','Extensive']
access_cyber = df.groupby('Data_Access_Level')['Score_Cyber'].mean().reindex(access_order)
bars = ax.bar(access_order, access_cyber.values,
              color=plt.cm.RdYlGn_r(np.linspace(0.2,0.9,4)), width=0.6, edgecolor='white')
ax.set_title('Avg Cyber Risk Score by Data Access Level', fontweight='bold')
ax.set_ylabel('Avg Cyber Risk Score')
for bar, val in zip(bars, access_cyber.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')

# Compliance violations distribution
ax = axes[0,2]
viol_counts = df['Compliance_Violations'].value_counts().sort_index()
ax.bar(viol_counts.index, viol_counts.values, color='#9B59B6', edgecolor='white', width=0.6)
ax.set_title('Compliance Violations Distribution', fontweight='bold')
ax.set_xlabel('Number of Violations')
ax.set_ylabel('Number of Vendors')

# Credit rating distribution
ax = axes[1,0]
rating_order = ['AAA','AA','A','BBB','BB','B','CCC']
rating_counts = df['Credit_Rating'].value_counts().reindex(rating_order)
bar_colors = plt.cm.RdYlGn(np.linspace(0.9,0.1,7))
bars = ax.bar(rating_order, rating_counts.values, color=bar_colors, width=0.6, edgecolor='white')
ax.set_title('Vendor Credit Rating Distribution', fontweight='bold')
ax.set_ylabel('Number of Vendors')
for bar, val in zip(bars, rating_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            str(val), ha='center', fontsize=9, fontweight='bold')

# Concentration risk scatter
ax = axes[1,1]
for tier, color in RISK_COLORS.items():
    sub = df[df['Risk_Tier']==tier]
    ax.scatter(sub['Pct_Category_Spend'], sub['Annual_Spend_USD'],
               color=color, s=60, alpha=0.8, label=tier,
               edgecolors='white', linewidth=0.8, zorder=3)
ax.set_title('Concentration Risk: Category Share vs Spend', fontweight='bold')
ax.set_xlabel('% of Category Spend')
ax.set_ylabel('Annual Spend (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
ax.legend(fontsize=8)

# SLA breaches vs risk tier
ax = axes[1,2]
sla_tier = df.groupby('Risk_Tier')['SLA_Breaches_YTD'].mean().reindex(tier_order)
bars = ax.bar(tier_order, sla_tier.values, color=[RISK_COLORS[t] for t in tier_order],
              width=0.55, edgecolor='white')
ax.set_title('Avg SLA Breaches YTD by Risk Tier', fontweight='bold')
ax.set_ylabel('Avg SLA Breaches')
for bar, val in zip(bars, sla_tier.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig
plt.show()


## 5. Tier-Based Due Diligence Checklist Generator

Each risk tier triggers a different depth of due diligence — mirroring the **risk-tiered approach mandated by OCC and Basel third-party risk guidance**, where the intensity of oversight scales with the residual risk of the relationship.


In [ ]:
DUE_DILIGENCE_CHECKLISTS = {
    'Critical': [
        'On-site audit conducted within last 12 months',
        'SOC 2 Type II report obtained and reviewed by Risk Committee',
        'Board-level approval required for contract renewal',
        'Business continuity / disaster recovery plan reviewed and tested',
        'Financial statements reviewed quarterly by Finance team',
        'Concentration risk mitigation plan documented (backup vendor identified)',
        'Cyber insurance certificate of currency obtained',
        'Right-to-audit clause confirmed in contract',
        'Exit plan / transition strategy documented',
        'Quarterly risk review scheduled with vendor management committee',
    ],
    'High': [
        'Annual on-site or virtual audit conducted',
        'Security certification (ISO 27001 or SOC 2) verified',
        'Senior management approval required for contract renewal',
        'Financial health reviewed semi-annually',
        'Data processing agreement reviewed (if applicable)',
        'Incident response plan requested and reviewed',
        'Semi-annual risk review scheduled',
    ],
    'Medium': [
        'Annual self-assessment questionnaire completed by vendor',
        'Standard contract terms reviewed by Legal',
        'Financial health reviewed annually',
        'Basic security questionnaire completed',
        'Annual risk review scheduled',
    ],
    'Low': [
        'Standard onboarding questionnaire completed',
        'Basic contract terms confirmed',
        'Annual relationship review (light-touch)',
    ],
}

print("DUE DILIGENCE REQUIREMENTS BY RISK TIER")
print("=" * 70)
for tier in ['Critical','High','Medium','Low']:
    count = (df['Risk_Tier']==tier).sum()
    print(f"\n{tier.upper()} RISK TIER — {count} vendors")
    print("-" * 70)
    for item in DUE_DILIGENCE_CHECKLISTS[tier]:
        print(f"  [ ] {item}")

# Apply checklist reference to register
df['DD_Checklist_Items'] = df['Risk_Tier'].map(lambda t: len(DUE_DILIGENCE_CHECKLISTS[t]))
df['DD_Frequency'] = df['Risk_Tier'].map({
    'Critical':'Quarterly','High':'Semi-Annual','Medium':'Annual','Low':'Biennial'})

print()
print()
print("Sample vendor due diligence assignment:")
print(df[['Vendor_Name','Risk_Tier','DD_Checklist_Items','DD_Frequency']].head(10).to_string(index=False))


## 6. Operational Risk Management Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('TPRM Operational Dashboard — Management Summary', fontsize=15, fontweight='bold')

# Top 15 vendors by risk score
ax = axes[0,0]
top15 = df.nlargest(15, 'Adjusted_Risk_Score')
bar_colors = [RISK_COLORS[t] for t in top15['Risk_Tier']]
ax.barh(top15['Vendor_Name'][::-1], top15['Adjusted_Risk_Score'][::-1],
        color=bar_colors[::-1], edgecolor='white')
ax.set_title('Top 15 Highest Risk Vendors', fontweight='bold')
ax.set_xlabel('Adjusted Risk Score')

# Due diligence frequency workload
ax = axes[0,1]
dd_freq = df['DD_Frequency'].value_counts().reindex(['Quarterly','Semi-Annual','Annual','Biennial'])
bars = ax.bar(dd_freq.index, dd_freq.values, color=PALETTE[:4], width=0.6, edgecolor='white')
ax.set_title('Due Diligence Review Frequency Workload', fontweight='bold')
ax.set_ylabel('Number of Vendors')
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, dd_freq.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            str(val), ha='center', fontsize=10, fontweight='bold')

# Sole source critical vendors
ax = axes[0,2]
sole_source_critical = df[(df['Is_Sole_Source']) & (df['Contract_Criticality']=='Critical')]
sole_source_other     = df[~((df['Is_Sole_Source']) & (df['Contract_Criticality']=='Critical'))]
bars = ax.bar(['Sole Source +\nCritical Contract','All Other\nVendors'],
              [len(sole_source_critical), len(sole_source_other)],
              color=['#E74C3C','#3BAB6F'], width=0.5, edgecolor='white')
ax.set_title('Single Points of Failure\n(Sole-Source Critical Vendors)', fontweight='bold')
ax.set_ylabel('Number of Vendors')
for bar, val in zip(bars, [len(sole_source_critical), len(sole_source_other)]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            str(val), ha='center', fontsize=11, fontweight='bold')

# Risk score vs spend bubble matrix
ax = axes[1,0]
for tier, color in RISK_COLORS.items():
    sub = df[df['Risk_Tier']==tier]
    ax.scatter(sub['Annual_Spend_USD'], sub['Adjusted_Risk_Score'],
               s=sub['SLA_Breaches_YTD']*40+30, color=color, alpha=0.75,
               label=tier, edgecolors='white', linewidth=0.8, zorder=3)
ax.axhline(65, color='#E74C3C', linestyle='--', linewidth=1.2, label='Critical threshold')
ax.axhline(45, color='#F39C12', linestyle='--', linewidth=1.2, label='High threshold')
ax.set_title('Vendor Risk Matrix\n(bubble size = SLA breaches)', fontweight='bold')
ax.set_xlabel('Annual Spend (USD)')
ax.set_ylabel('Adjusted Risk Score')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
ax.legend(fontsize=7)

# KPI Summary table
ax = axes[1,1]
ax.axis('off')
critical_n = (df['Risk_Tier']=='Critical').sum()
high_n     = (df['Risk_Tier']=='High').sum()
total_spend= df['Annual_Spend_USD'].sum()
crit_spend = df[df['Risk_Tier']=='Critical']['Annual_Spend_USD'].sum()
breach_n   = df['Recent_Breach'].sum()
no_cert_n  = ((~df['Has_ISO27001']) & (~df['Has_SOC2'])).sum()
summary_data = [
    ['Metric','Value'],
    ['Total Vendors', str(len(df))],
    ['Critical Risk Tier', f'{critical_n} ({critical_n/len(df)*100:.0f}%)'],
    ['High Risk Tier', f'{high_n} ({high_n/len(df)*100:.0f}%)'],
    ['Total Annual Spend', f'\${total_spend/1e6:.1f}M'],
    ['Spend in Critical Tier', f'\${crit_spend/1e6:.1f}M'],
    ['Vendors w/ Recent Breach', str(breach_n)],
    ['Vendors w/ No Certification', str(no_cert_n)],
    ['Sole-Source Critical', str(len(sole_source_critical))],
    ['High-Risk Country Vendors', str(df['Is_High_Risk_Country'].sum())],
    ['Avg Risk Score', f"{df['Adjusted_Risk_Score'].mean():.1f}/100"],
]
table = ax.table(cellText=summary_data[1:], colLabels=summary_data[0],
                 cellLoc='left', loc='center', colWidths=[0.65,0.35])
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 1.7)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1A1A2E'); cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#F8F9FA')
ax.set_title('TPRM Portfolio KPI Summary', fontweight='bold', pad=20)

# Risk tier trend by onboarding year
ax = axes[1,2]
df['Onboard_Year'] = df['Onboarded_Date'].dt.year
year_tier = df.groupby(['Onboard_Year','Risk_Tier']).size().unstack(fill_value=0)
year_tier = year_tier.reindex(columns=tier_order, fill_value=0)
year_tier.plot(kind='bar', stacked=True, ax=ax,
              color=[RISK_COLORS[t] for t in tier_order], edgecolor='white', width=0.7)
ax.set_title('Vendor Onboarding by Year & Risk Tier', fontweight='bold')
ax.set_xlabel('Onboarding Year')
ax.set_ylabel('Number of Vendors')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Risk Tier', fontsize=7)

plt.tight_layout()
plt.savefig('/home/claude/operational_dashboard.png', bbox_inches='tight')
plt.show()
